In [0]:
# Installation des librairies de base
%pip install langgraph langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Mise à jour forcée pour éviter le conflit Pydantic/Typing (l'erreur de tout à l'heure)
%pip install -U typing_extensions pydantic langchain-core

# Redémarrage du moteur pour prendre en compte les installations
dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os

# 1. Configuration et imports
sys.path.append(os.path.abspath('..'))
from src.interview_graph import build_interview_graph
try:
    hf_token = dbutils.secrets.get(scope="llm_secrets", key="huggingface_token")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    print(" Token HuggingFace récupéré et injecté dans l'environnement !")
except Exception as e:
    print(f"Erreur avec le secret Databricks : {e}")
# 2. Paramètres de l'entretien
sujet = "SQL et Bases de Données"
nb_questions = 3

print(f" Initialisation de l'entretien sur : {sujet}")
app = build_interview_graph()

# --- PHASE 1 : GÉNÉRATION DES QUESTIONS ---
etat_initial = {
    "sujet_entretien": sujet,
    "nombre_questions": nb_questions,
    "questions": [],
    "reponses": [],
    "contextes_rag": [],
    "feedback_recruteur": "",
    "passed_recruteur": False,
    "verdict_juge": "",
    "juge_est_daccord": True
}

# L'Agent 1 génère les questions
resultat_gen = app.invoke(etat_initial)
questions_posees = resultat_gen["questions"]

# --- PHASE 2 : INTERACTION LIVE (INPUT) ---
reponses_utilisateur = []

print("\n" + "="*50)
print(" L'ENTRETIEN COMMENCE MAINTENANT")
print("="*50 + "\n")

for i, q in enumerate(questions_posees):
    print(f" QUESTION {i+1} : {q}")
    
    # C'est ici que tu vas répondre dans l'interface Databricks
    reponse = input(f"Votre réponse à la Q{i+1} > ")
    reponses_utilisateur.append(reponse)
    print(f"✅ Réponse enregistrée.\n")

# --- PHASE 3 : ÉVALUATION GLOBALE ---
print(" Analyse de vos réponses par le Recruteur et le Juge...")

etat_final = {
    **resultat_gen,
    "reponses": reponses_utilisateur
}
config = {"recursion_limit": 10}
# Le graphe reprend son cours : RAG -> Recruteur (Éval) -> Juge
verdict_final = app.invoke(etat_final, config=config)

# --- AFFICHAGE DES RÉSULTATS ---
print("\n" + "="*50)
print(" RÉSULTAT DE L'ENTRETIEN")
print("="*50)

print(f"\n BILAN DU RECRUTEUR :")
print(verdict_final["feedback_recruteur"])

print(f"\n DÉCISION : {' ADMIS' if verdict_final['passed_recruteur'] else ' RECALÉ'}")

print(f"\n AVIS DU JUGE :")
print(verdict_final["verdict_juge"])